In [5]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Fairness_Aware_Explainable_RL


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/.shortcut-targets-by-id/1vXU9MfjUMNHQrwySqrUuPfBor5G5ARpQ/Fairness_Aware_Explainable_RL


In [ ]:
# !unzip -q "/content/drive/MyDrive/Fairness_Aware_Explainable_RL.zip" -d /content/
# %cd /content/Fairness_Aware_Explainable_RL

In [6]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

True
Tesla T4


In [ ]:
# !python baselines/sasrec_baseline_fairir_gpu.py

!python rl_model/faircsrec.py --fairness_lambda 0.0


Traceback (most recent call last):
  File "/content/drive/.shortcut-targets-by-id/1vXU9MfjUMNHQrwySqrUuPfBor5G5ARpQ/Fairness_Aware_Explainable_RL/rl_model/faircsrec.py", line 76, in <module>
    from metrics.user_fairness_metrics import (load_user_gender,
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1331, in _find_and_load_unlocked
  File "<frozen importlib._bootstrap>", line 935, in _load_unlocked
  File "<frozen importlib._bootstrap_external>", line 995, in exec_module
  File "<frozen importlib._bootstrap_external>", line 1091, in get_code
  File "<frozen importlib._bootstrap_external>", line 1191, in get_data
KeyboardInterrupt
^C


In [ ]:
!python baselines/fairsasrec.py --fairness_lambda 0.3


Device: cpu
Users: 6034 | Items: 3125
Gender loaded — Male: 4326 | Female: 1708
object address  : 0x79b84b25c700
object refcount : 3
object type     : 0xa284e0
object type name: KeyboardInterrupt
object repr     : KeyboardInterrupt()
lost sys.stderr
^C


In [ ]:
!mkdir -p results/FairSASRec-ml1m && cp results/FairSASRec/fairsasrec_best.pt results/FairSASRec-ml1m/

In [ ]:
# !pip install cornac -q

!python tests/evaluate_final.py

Device: cpu

--- BPR ---
  HR@10 0.6613 | NDCG@10 0.3823 | DP 0.5909 | EO 0.8489 | Gini 0.8733 | Cov 0.4976 | JS@20 0.0849

--- SASRec ---
  HR@10 0.1783 | NDCG@10 0.0874 | DP 0.5644 | EO 0.9700 | Gini 0.8952 | Cov 0.4368 | JS@20 0.0188

--- FairSASRec (results/FairSASRec-ml1m/fairsasrec_final.pt) ---
  HR@10 0.1843 | NDCG@10 0.0908 | DP 0.5347 | EO 0.8823 | Gini 0.8816 | Cov 0.4330 | JS@20 0.0173

PAPER TABLE (ml-1m, leave-one-out | acc: 100-neg | fair: full-rank)
Model          HR@10  NDCG@10   DP@10   EO@10    Gini    Cov   JS@20   JS@50
BPR           0.6613   0.3823  0.5909  0.8489  0.8733  0.498  0.0849  0.0646
SASRec        0.1783   0.0874  0.5644  0.9700  0.8952  0.437  0.0188  0.0101
FairSASRec    0.1843   0.0908  0.5347  0.8823  0.8816  0.433  0.0173  0.0097
--------------------------------------------------------------------------
FairIR published (their protocol): HR .2833 NDCG .2777 DP .5741 EO .6258 | JS@20 .1301 JS@50 .0968
Saved: results/final/paper_table.json


In [ ]:
import torch, glob
for p in sorted(glob.glob('results/**/*.pt', recursive=True)):
    try:
        s = torch.load(p, map_location='cpu')
        emb = tuple(s['item_emb.weight'].shape) if 'item_emb.weight' in s else '?'
        pos = tuple(s['pos_emb.weight'].shape) if 'pos_emb.weight' in s else '?'
        fair = 'fairness_alpha' in s
        print(f"{str(emb):<12} pos={str(pos):<10} fair={fair}  {p}")
    except Exception as e:
        print("ERR", p)

(3126, 64)   pos=(101, 64)  fair=False  results/FairCSRec/lambda_0.0/faircsrec_best.pt
(1009, 64)   pos=(11, 64)   fair=True  results/FairSASRec-100k/fairsasrec_best.pt
(1009, 64)   pos=(11, 64)   fair=True  results/FairSASRec-100k/fairsasrec_final.pt
(1009, 64)   pos=(21, 64)   fair=True  results/FairSASRec-ml1m/fairsasrec_best.pt
(3126, 64)   pos=(21, 64)   fair=True  results/FairSASRec-ml1m/fairsasrec_final.pt
(1009, 64)   pos=(21, 64)   fair=True  results/FairSASRec/fairsasrec_best.pt
(1009, 64)   pos=(21, 64)   fair=True  results/FairSASRec/fairsasrec_final.pt
(3126, 64)   pos=(101, 64)  fair=False  results/SASRec-fairir-v2/sasrec_best.pt
(3126, 64)   pos=(101, 64)  fair=False  results/SASRec-fairir-v2/sasrec_final.pt
(3126, 64)   pos=(21, 64)   fair=False  results/SASRec-fairir/sasrec_best.pt
(3126, 64)   pos=(21, 64)   fair=False  results/SASRec-fairir/sasrec_final.pt
(1009, 64)   pos=(21, 64)   fair=False  results/SASRec-ml-100k-fairir/sasrec_best.pt
(1009, 64)   pos=(21, 64)  

In [ ]:
!python -c " import json m1 = json.load(open('data/ml-1m/meta.json')) m2 = json.load(open('data/ml-1m-fairir/meta.json')) print('n_items ml-1m:', m1['n_items']) print('n_items ml-1m-fairir:', m2['n_items']) print('Same item2idx?', m1.get('item2idx') == m2.get('item2idx')) print('Same user2idx?', m1.get('user2idx') == m2.get('user2idx')) "

In [7]:
!python make_explanation_figure.py --run p1 --dataset ml-1m

Device: cuda | dataset: ml-1m
  FairSASRec: results/FairSASRec-ml1m/fairsasrec_final.pt
  SASRec    : results/SASRec-200/sasrec_best.pt

Computing gender exposure gaps from the backbone's own lists...
Comparing top-1 recommendations across all test users...

Top-1 changed for 5146 / 6034 users (85.3%)
  mean gender gap, displaced item : 0.4568
  mean gender gap, replacement    : 0.1029
  mean reduction                  : +0.3538 (-77.5%)
  paired t-test                   : t=93.13, p=0.00e+00
  replacement less skewed in      : 88.3% of cases
Saved runs/p1/final/explanation_stats.json
Saved runs/p1/figures/fig5_explanation.png
